### Gold: `03_gold.window_grid` — 20-minute wall-clock windows per operational day
A window is in the grid only if at least one real instant falls in it (DST gap on 2025-03-09 02:00–03:00).
Widget `settings` = `spec` (default) or `conventional` (Q4, writes `window_grid_conventional`).

In [0]:
%run ../../config/config

In [0]:
%run ../../utils/zsi_functions

In [0]:
dbutils.widgets.text("settings", "spec")
settings_name = dbutils.widgets.get("settings")
s = ZSI_SETTINGS[settings_name]
print(settings_name, s)

In [0]:
from pyspark.sql.functions import to_date, to_timestamp

grid = (spark.createDataFrame(build_grid(s), GRID_COLUMNS)
        .withColumn("operational_date", to_date("operational_date"))
        .withColumn("window_start", to_timestamp("window_start")))

grid.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("03_gold", f"window_grid{s['table_suffix']}"))

In [0]:
display(spark.sql(f"""
SELECT operational_date, COUNT(*) AS wall_clock_windows, SUM(CAST(window_exists AS INT)) AS windows_in_grid,
       SUM(CAST(is_am AS INT)) AS morning_windows
FROM {tbl('03_gold', f"window_grid{s['table_suffix']}")}
GROUP BY 1 ORDER BY 1"""))